### imports

In [10]:
import pandas as pd
import numpy as np
import anthropic
import json
import time
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
from concurrent.futures import ThreadPoolExecutor

### setup

In [ ]:
ANTHROPIC_API_KEY = "ANTHROPIC_API_KEY"
MODEL = "claude-sonnet-4-6"
N = 15
MAX_SAMPLES = 50
NUM_PARALLEL = 3

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """You are an NFL prediction market expert.
Given a window of Kalshi prices and Reddit comments, predict if the price will go UP, DOWN, or FLAT in the next minute.
Always use the make_prediction tool to submit your answer."""

USER_PROMPT = """Game: {game}
Market: {kalshi_title}
Minute: {offset}

History (last {N} minutes):
{window_text}

What is the next move?"""

TOOL = {
    "name": "make_prediction",
    "description": "Submit prediction for the next minute",
    "input_schema": {
        "type": "object",
        "properties": {
            "label": {"type": "string", "enum": ["UP", "DOWN", "FLAT"]},
            "reasoning": {"type": "string"}
        },
        "required": ["label"]
    }
}

### prediction

In [30]:
def build_window_text(window_df):
    lines = []
    for i, (_, row) in enumerate(window_df.iterrows()):
        t_label = 'now' if i == len(window_df) - 1 else f't-{len(window_df) - 1 - i}'
        comments = " | ".join(c[:100] for c in list(row['comments_list'])[:3])
        lines.append(f"[{t_label:>4}] Price: {row['price_close']:.3f} | Comments: {comments or 'N/A'}")
    return "\n".join(lines)

def load_samples(path, n_back):
    df = pd.read_parquet(path).dropna(subset=['delta'])
    games = df['game_id'].unique()
    np.random.seed(42)
    np.random.shuffle(games)
    test_games = games[int(0.85 * len(games)):]
    test_df = df[df['game_id'].isin(test_games)].copy()
    
    samples = []
    for _, g_df in test_df.groupby('game_id'):
        g_df = g_df.sort_values('timestamp').reset_index(drop=True)
        if len(g_df) <= n_back: continue
        for i in range(n_back, len(g_df)):
            samples.append({
                'game': g_df.iloc[i]['game'],
                'kalshi_title': g_df.iloc[i]['kalshi_title'],
                'offset': g_df.iloc[i]['offset_from_kickoff_min'],
                'label': g_df.iloc[i]['label'],
                'window': g_df.iloc[i-n_back : i+1]
            })
    return samples

all_samples = load_samples('data.parquet', N)
indices = np.random.choice(len(all_samples), min(MAX_SAMPLES, len(all_samples)), replace=False)
samples = [all_samples[i] for i in sorted(indices)]
print(f"Loaded {len(samples)} samples")

Loaded 50 samples


In [31]:
def predict(sample):
    txt = build_window_text(sample['window'])
    prompt = USER_PROMPT.format(
        game=sample['game'],
        kalshi_title=sample['kalshi_title'],
        offset=sample['offset'],
        N=N,
        window_text=txt
    )
    
    try:
        res = client.messages.create(
            model=MODEL,
            max_tokens=512,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
            tools=[TOOL],
            tool_choice={"type": "auto"}
        )
        
        for b in res.content:
            if b.type == "tool_use":
                return b.input['label'], b.input.get('reasoning', '')
        
        full_text = " ".join(b.text for b in res.content if b.type == "text").upper()
        for l in ["UP", "DOWN", "FLAT"]:
            if l in full_text: return l, "(text fallback)"
            
    except Exception as e:
        print(f"Error: {e}")
    return None, None

In [32]:
results = []
def process_sample(s):
    p, r = predict(s)
    return {'true': s['label'], 'pred': p, 'reason': r}

with ThreadPoolExecutor(max_workers=NUM_PARALLEL) as pool:
    results = list(tqdm(pool.map(process_sample, samples), total=len(samples)))

results_df = pd.DataFrame(results)

100%|██████████| 50/50 [02:50<00:00,  3.40s/it]


### analysis of results

In [33]:
final_df = results_df.dropna(subset=['pred'])
print(f"Final Accuracy: {accuracy_score(final_df['true'], final_df['pred']):.3f}")
print("\nClassification Report:")
print(classification_report(final_df['true'], final_df['pred']))

Final Accuracy: 0.520

Classification Report:
              precision    recall  f1-score   support

        DOWN       0.33      0.23      0.27        13
        FLAT       0.68      0.81      0.74        26
          UP       0.20      0.18      0.19        11

    accuracy                           0.52        50
   macro avg       0.40      0.41      0.40        50
weighted avg       0.48      0.52      0.50        50



In [34]:
correct = final_df[final_df['true'] == final_df['pred']].head(3)
incorrect = final_df[final_df['true'] != final_df['pred']].head(3)

print("CORRECT SAMPLES (Actual | Claude | Reason)")
for _, r in correct.iterrows():
    print(f"{r['true']} | {r['pred']} | {str(r['reason'])[:100]}...")

print("\nINCORRECT SAMPLES (Actual | Claude | Reason)")
for _, r in incorrect.iterrows():
    print(f"{r['true']} | {r['pred']} | {str(r['reason'])[:100]}...")


CORRECT SAMPLES (Actual | Claude | Reason)
FLAT | FLAT | The price has been locked at 0.690 for 12 consecutive minutes with no signs of movement. Reddit sent...
FLAT | FLAT | Price has been locked at 0.650 for 14 of the last 15 minutes with virtually no volatility. We're 6 m...
FLAT | FLAT | The Saints' price has dropped sharply from 0.260 to 0.180 in just 2 minutes following a near-turnove...

INCORRECT SAMPLES (Actual | Claude | Reason)
DOWN | FLAT | Price has been consolidating between 0.89-0.96 for the past several minutes, indicating the market h...
FLAT | UP | Price has shown a clear upward trend over the last 3 minutes (0.610 → 0.640 → 0.650). Recent comment...
FLAT | UP | Strong upward price momentum from 0.640 to 0.790 over the last ~10 minutes. SF appears to be pulling...
